In [ ]:
from constants import *
from astropy import units as u
from phoenix_grid_creator.spectral_grid import spectral_grid
from spectrum_component_analyser.phoenix_spectrum import phoenix_spectrum
from matplotlib import pyplot as plt
import numpy as np
from scipy.ndimage import gaussian_filter1d
from spectrum_component_analyser.spectrum import DEFAULT_FLUX_UNIT
from pathlib import Path
from astropy.visualization import quantity_support
import spectres

quantity_support()

x_min, x_max = 0.85 * u.um, 0.86 * u.um
# x_min, x_max = 1.6 * u.um, 1.8 * u.um

fits_file_paths = list(Path(package_path / "raw_phoenix_spectra").rglob("*.fits"))

resolution = 0.01 * u.um

resolution : u.Quantity[u.K] = .01*u.um
observational_wavelengths : np.ndarray = np.arange(0.8, 5.3, resolution.value) * u.um
raw_spec_grid = spectral_grid.from_local_raw(files=fits_file_paths, observational_wavelengths=observational_wavelengths, parallelise=False)
raw_spec_grid.Wavelengths = raw_spec_grid.Wavelengths.to(u.um)
raw_spec_grid.Fluxes *= u.Jy

In [ ]:
example_spectrum = raw_spec_grid.get_spectrum(3500 * u.K, 0.0 * u.dex, 1.5 * u.dex).plot()

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(16, 12), sharex=True, sharey=True)

# plt.style.use('seaborn-v0_8-paper')
plt.rcParams.update({'font.size': 18})
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["STIXGeneral"],
    "mathtext.fontset": "stix",
})

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.close('all')

plt.rcParams.update({'font.size': 18})
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["STIXGeneral"],
    "mathtext.fontset": "stix",
})

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(8, 16))
ax1.set_title("$\log g = 4$ dex")
ax2.set_title("[Fe/H] = 0 dex")
ax3.set_title(r"$T_\mathrm{eff} = 3500$ K")

# Define your main plot limits first
ax1.set_xlim(1950, 5000)
ax1.set_ylim(-2.5, .5)

ax1.set_xlabel(r'$T_\text{eff}$ [K]')
ax1.set_ylabel('[Fe/H] [dex]')


parameter_coords = [
    (2500 * u.K, 0.0 * u.dex),
    (3500 * u.K, 0.0 * u.dex),
    (4500 * u.K, 0.0 * u.dex),
    (2500 * u.K, -1.0 * u.dex),
    (3500 * u.K, -1.0 * u.dex),
    (4500 * u.K, -1.0 * u.dex),
    (2500 * u.K, -2.0 * u.dex),
    (3500 * u.K, -2.0 * u.dex),
    (4500 * u.K, -2.0 * u.dex),
]

# This forces the renderer to initialize the main axes' bounding box
fig.canvas.draw()

for i, (T_eff, FeH) in enumerate(parameter_coords):
    print(f"{i} START")
    
    x_val = T_eff.value
    y_val = FeH.value

    # We define the inset size in the coordinate system of the main plot
    # Example: width is 400 Kelvin wide, height is 0.3 dex tall
    width = 800 
    height = .75
    
    # Modern method: axes.inset_axes([x, y, width, height], transform=ax.transData)
    ax_sub = ax1.inset_axes(
        [x_val - width/2, y_val - height/2, width, height],
        transform=ax1.transData
    )
    
    Log_g = 4 * u.dex
    spec = raw_spec_grid.get_spectrum(T_eff, FeH, Log_g)

    # Convert to pure numpy arrays to avoid rendering unit overhead
    waves = spec.Wavelengths.value if hasattr(spec.Wavelengths, 'value') else spec.Wavelengths
    fluxes = spec.Fluxes.value if hasattr(spec.Fluxes, 'value') else spec.Fluxes

    # Plot and force limits
    ax_sub.plot(waves[::10], fluxes[::10],  lw=1)
    ax_sub.set_xlim(waves.min(), waves.max())
    ax_sub.set_ylim(fluxes.min(), fluxes.max())
    if i == 0:
        ax_sub.set_xlabel(r"Wavelength [$\mathrm{\mu m}]$", fontsize=12)
        ax_sub.set_ylabel("Flux [Jy]", fontsize=12)
    
    # Clean up inset aesthetics
    ax_sub.set_xticks([])
    ax_sub.set_yticks([])
    ax_sub.patch.set_alpha(0.7)
    
    print(f"{i} DONE")

import matplotlib.pyplot as plt
import numpy as np

# Define your main plot limits first
ax2.set_xlim(2000, 5000)
ax2.set_ylim(0, 6)

ax2.set_xlabel(r'$T_\text{eff}$ [K]')
ax2.set_ylabel(r'$\log g$ [dex]')


parameter_coords = [
    (2500 * u.K, 1.0 * u.dex),
    (3500 * u.K, 1.0 * u.dex),
    (4500 * u.K, 1.0 * u.dex),
    (2500 * u.K, 3.0 * u.dex),
    (3500 * u.K, 3.0 * u.dex),
    (4500 * u.K, 3.0 * u.dex),
    (2500 * u.K, 5.0 * u.dex),
    (3500 * u.K, 5.0 * u.dex),
    (4500 * u.K, 5.0 * u.dex),
]

# This forces the renderer to initialize the main axes' bounding box
fig.canvas.draw()

for i, (T_eff, Log_g) in enumerate(parameter_coords):
    print(f"{i} START")
    
    x_val = T_eff.value
    y_val = Log_g.value

    # We define the inset size in the coordinate system of the main plot
    # Example: width is 400 Kelvin wide, height is 0.3 dex tall
    width = 800
    height = 1.5
    
    # Modern method: axes.inset_axes([x, y, width, height], transform=ax.transData)
    ax_sub = ax2.inset_axes(
        [x_val - width/2, y_val - height/2, width, height],
        transform=ax2.transData
    )
    
    FeH = 0 * u.dex
    spec = raw_spec_grid.get_spectrum(T_eff, FeH, Log_g)

    # Convert to pure numpy arrays to avoid rendering unit overhead
    waves = spec.Wavelengths.value if hasattr(spec.Wavelengths, 'value') else spec.Wavelengths
    fluxes = spec.Fluxes.value if hasattr(spec.Fluxes, 'value') else spec.Fluxes

    # Plot and force limits
    ax_sub.plot(waves[::10], fluxes[::10],  lw=1)
    ax_sub.set_xlim(waves.min(), waves.max())
    ax_sub.set_ylim(fluxes.min(), fluxes.max())
    
    # Clean up inset aesthetics
    ax_sub.set_xticks([])
    ax_sub.set_yticks([])
    ax_sub.patch.set_alpha(0.7)
    
    print(f"{i} DONE")



import matplotlib.pyplot as plt
import numpy as np

# Define your main plot limits first
ax3.set_xlim(-2.5 * u.dex, 0.5 * u.dex)
ax3.set_ylim(0 * u.dex, 6 * u.dex)

ax3.set_xlabel(r'[Fe/H] [dex]')
ax3.set_ylabel(r'$\log g$ [dex]')


parameter_coords = [
    (0.0 * u.dex, 1.0 * u.dex),
    (-1.0 * u.dex, 1.0 * u.dex),
    (-2.0 * u.dex, 1.0 * u.dex),
    (0.0 * u.dex, 3.0 * u.dex),
    (-1.0 * u.dex, 3.0 * u.dex),
    (-2.0 * u.dex, 3.0 * u.dex),
    (0.0 * u.dex, 5.0 * u.dex),
    (-1.0 * u.dex, 5.0 * u.dex),
    (-2.0 * u.dex, 5.0 * u.dex),
]

# This forces the renderer to initialize the main axes' bounding box
fig.canvas.draw()

for i, (FeH, Log_g) in enumerate(parameter_coords):
    print(f"{i} START")
    
    x_val = FeH.value
    y_val = Log_g.value

    # We define the inset size in the coordinate system of the main plot
    # Example: width is 400 Kelvin wide, height is 0.3 dex tall
    width = .8
    height = 1.5
    
    # Modern method: axes.inset_axes([x, y, width, height], transform=ax.transData)
    ax_sub = ax3.inset_axes(
        [x_val - width/2, y_val - height/2, width, height],
        transform=ax3.transData
    )
    
    T_eff = 3500 * u.K
    spec = raw_spec_grid.get_spectrum(T_eff, FeH, Log_g)

    # Convert to pure numpy arrays to avoid rendering unit overhead
    waves = spec.Wavelengths.value if hasattr(spec.Wavelengths, 'value') else spec.Wavelengths
    fluxes = spec.Fluxes.value if hasattr(spec.Fluxes, 'value') else spec.Fluxes

    # Plot and force limits
    ax_sub.plot(waves[::10], fluxes[::10],  lw=1)
    ax_sub.set_xlim(waves.min(), waves.max())
    ax_sub.set_ylim(fluxes.min(), fluxes.max())
    
    # Clean up inset aesthetics
    ax_sub.set_xticks([])
    ax_sub.set_yticks([])
    ax_sub.patch.set_alpha(0.7)
    
    print(f"{i} DONE")














plt.tight_layout()
plt.savefig("basis_for_degeneracies.svg", bbox_inches='tight', pad_inches=0.5)

plt.show()

- maybe add the 1 fixed parameter for each plot to the plot title? or just to the caption here? can add left/middle/right or top/middle/bottom text in the caption ig